# Predict

Run prediction using the model output from [02_modelling.ipynb](./02_modelling.ipynb)

Output: [nisjedata-substrat-xgbclassifier_norge_latest_25833.geojson](https://storage.googleapis.com/niva-geodata/MarintNaturKart/nisjedata-substrat-xgbclassifier_norge_latest_25833.geojson)

Comparison map [here](https://terriamap.p.niva.no/#start=%7B%22version%22%3A%228.0.0%22%2C%22initSources%22%3A%5B%7B%22stratum%22%3A%22user%22%2C%22models%22%3A%7B%22__User-Added_Data__%22%3A%7B%22isOpen%22%3Atrue%2C%22members%22%3A%5B%228245f6bc-c3d0-4fcc-aeaa-10faad0a3ec1%22%5D%2C%22knownContainerUniqueIds%22%3A%5B%22%2F%22%5D%2C%22type%22%3A%22group%22%7D%2C%22%2F%2FMarint+Natur+Kart%2FTest+Hard%2FL%C3%B8s%2FBlanding+Prediksjon+-+Norge%22%3A%7B%22splitDirection%22%3A1%2C%22knownContainerUniqueIds%22%3A%5B%22%2F%2FMarint+Natur+Kart%22%5D%2C%22type%22%3A%22wfs%22%7D%2C%22%2F%2FMarint+Natur+Kart%2FBunntypeklasser+for+Prediksjon+Norge%22%3A%7B%22splitDirection%22%3A-1%2C%22knownContainerUniqueIds%22%3A%5B%22%2F%2FMarint+Natur+Kart%22%5D%2C%22type%22%3A%22wfs%22%7D%2C%228245f6bc-c3d0-4fcc-aeaa-10faad0a3ec1%22%3A%7B%22splitSourceItemId%22%3A%22%2F%2FMarint+Natur+Kart%2FTest+Hard%2FL%C3%B8s%2FBlanding+Prediksjon+-+Norge%22%2C%22dereferenced%22%3A%7B%22name%22%3A%22Test+Hard%2FL%C3%B8s%2FBlanding+Prediksjon+-+Norge+%28copy%29%22%2C%22splitDirection%22%3A-1%7D%2C%22knownContainerUniqueIds%22%3A%5B%22__User-Added_Data__%22%5D%2C%22type%22%3A%22split-reference%22%7D%2C%22%2F%22%3A%7B%22type%22%3A%22group%22%7D%2C%22%2F%2FMarint+Natur+Kart%22%3A%7B%22knownContainerUniqueIds%22%3A%5B%22%2F%22%5D%2C%22type%22%3A%22group%22%7D%7D%2C%22workbench%22%3A%5B%22%2F%2FMarint+Natur+Kart%2FTest+Hard%2FL%C3%B8s%2FBlanding+Prediksjon+-+Norge%22%2C%22%2F%2FMarint+Natur+Kart%2FBunntypeklasser+for+Prediksjon+Norge%22%5D%2C%22timeline%22%3A%5B%22%2F%2FMarint+Natur+Kart%2FTest+Hard%2FL%C3%B8s%2FBlanding+Prediksjon+-+Norge%22%5D%2C%22initialCamera%22%3A%7B%22west%22%3A-1.8237304687500002%2C%22south%22%3A60.31062731740045%2C%22east%22%3A18.665771484375004%2C%22north%22%3A65.29346780107583%7D%2C%22homeCamera%22%3A%7B%22west%22%3A4%2C%22south%22%3A57.00000000000001%2C%22east%22%3A32%2C%22north%22%3A72%7D%2C%22viewerMode%22%3A%222d%22%2C%22showSplitter%22%3Atrue%2C%22splitPosition%22%3A0.5%2C%22settings%22%3A%7B%22baseMaximumScreenSpaceError%22%3A2%2C%22useNativeResolution%22%3Afalse%2C%22alwaysShowTimeline%22%3Afalse%2C%22baseMapId%22%3A%22basemap-openstreetmap%22%2C%22terrainSplitDirection%22%3A0%2C%22depthTestAgainstTerrainEnabled%22%3Afalse%7D%2C%22stories%22%3A%5B%5D%7D%5D%7D)

In [1]:
from pathlib import Path

import geopandas as gpd
import geoutils as gu
import numpy as np
import pandas as pd
import rasterio as rio
import xdem
from osgeo import gdal, ogr, osr

import subkart

In [2]:
res = subkart.features.RESOLUTION
nodata = 255
crs = "EPSG:25833"

In [3]:
classifier = subkart.utils.load_classifier()

# Predict for Norge

## Wave Exposure


In [4]:
bolge = subkart.sources.bolge_exposure()


## DEM 50

In [5]:
dem_norge = subkart.sources.dem_data()

## Predict

In [6]:
reverse_map = {v: k for k, v in subkart.labelling.BUNNTYPE_MAPPING.items()}
for region_name, region_list in subkart.sources.REGIONS.items():
    print(f"Processing region: {region_name}")
    gdf_sea_map_region = subkart.sources.sea_map_basisdata(region_list)
    gdf_sea_map_region = subkart.features.depth_preprocess(gdf_sea_map_region)
    transform, out_shape, bounds = subkart.features.to_raster_shapes(gdf_sea_map_region, res=res)
    dem_region = subkart.utils.resample_dem(dem_norge.crop(bounds), out_shape, transform, crs)
    print("Preparing bolge_region")
    bolge_region = bolge.reproject(
        crs=crs, res=res, bounds=dict(left=bounds[0], bottom=bounds[1], right=bounds[2], top=bounds[3])
    )
    print("Preparing X, valid_attrs, out_shape, transform")
    X, valid_attrs, out_shape, transform = subkart.features.build(
        dem_region, gdf_sea_map_region, bolge_region, valid_mask=None, res=res, dtype=np.float32
    )
    print("Predict ...")
    Y_pred = classifier.predict(X)

    pred_map = np.full(out_shape, nodata, dtype=np.uint8)
    pred_map[valid_attrs] = Y_pred.astype(np.uint8)

    # Mask nodata values explicitly to avoid the warning
    pred_map_masked = np.ma.masked_equal(pred_map, nodata)

    pred_raster = gu.Raster.from_array(pred_map_masked, transform=transform, crs=crs, nodata=nodata)
    fname = f"{region_name}_prediction.tif"
    pred_raster.to_file(Path(fname))
    print(f"Saved prediction for region {region_name} to {fname}\n")

Processing region: vestland


Preparing bolge_region


Preparing X, valid_attrs, out_shape, transform
Preparing sea_avg_depth...


/home/jovyan/marint-naturkart-niva/.pixi/envs/default/lib/python3.13/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


Preparing sea_avg_slope...


...

/home/jovyan/marint-naturkart-niva/.pixi/envs/default/lib/python3.13/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


Preparing wave exposure...


Stacking feature arrays...


Predict ...


Saved prediction for region nord to nord_prediction.tif



## Post processing

In [7]:
gdal.UseExceptions()

fname = subkart.utils.to_filename(
    f"nisjedata-substrat-{classifier.__class__.__name__.lower()}", "norge", "latest", crs.split(":")[1]
)
predict_file = f"{fname}.tif"

input_files = [f"{region_name}_prediction.tif" for region_name in subkart.sources.REGIONS.keys()]

subkart.utils.merge_rasters(input_files, predict_file, nodata=nodata)


In [8]:
subkart.vectorize.with_gdal(predict_file, "polygons_raw.gpkg", int(crs.split(":")[1]), nodata=255)

Polygons saved to polygons_raw.gpkg


In [9]:
gdf_raw = gpd.read_file("polygons_raw.gpkg")

In [10]:
fname = subkart.utils.to_filename(
    f"nisjedata-substrat-{classifier.__class__.__name__.lower()}", "norge", "latest", gdf_raw.crs.to_epsg()
)

reverse_map = {v: k for k, v in subkart.labelling.BUNNTYPE_MAPPING.items()}
gdf_raw["BunnType"] = gdf_raw["DN"].map(reverse_map)
gdf_raw.to_file(f"{fname}.gpkg", driver="GPKG", layer="bunntyper")
gdf_raw.to_parquet(f"{fname}.geo.parquet", compression="snappy")
subkart.utils.to_postgis(gdf_raw, fname)

NIVAGIS_CONNECTION_STR not set. Skipping PostGIS upload.
